# Лабораторная работа №1
## Пример конечного автомата на основе работы стиральной машины

Здор Матвей 342951 Р3324

Установка библиотеки

In [1]:
%pip install transitions

Note: you may need to restart the kernel to use updated packages.


Импорты

In [2]:
import transitions

from transitions import Machine

Класс стиральной машины с выходными сигналами (callbacks)

In [3]:
class WashingMachine:
    """Модель стиральной машины с выходными сигналами."""

    def on_enter_idle(self):
        print("  [ВЫХОД] Клапан: ВЫКЛ | Барабан: СТОП | Насос: ВЫКЛ | Индикатор: READY")

    def on_enter_filling(self):
        print("  [ВЫХОД] Клапан: ВКЛ  | Барабан: СТОП | Насос: ВЫКЛ | Индикатор: WORK")

    def on_enter_washing(self):
        print("  [ВЫХОД] Клапан: ВЫКЛ | Барабан: СТИРКА | Насос: ВЫКЛ | Индикатор: WORK")

    def on_enter_rinsing(self):
        print("  [ВЫХОД] Клапан: ВКЛ  | Барабан: ПОЛОСКАНИЕ | Насос: ВКЛ | Индикатор: WORK")

    def on_enter_spinning(self):
        print("  [ВЫХОД] Клапан: ВЫКЛ | Барабан: ОТЖИМ | Насос: ВКЛ | Индикатор: WORK")

    def on_enter_finished(self):
        print("  [ВЫХОД] Клапан: ВЫКЛ | Барабан: СТОП | Насос: ВЫКЛ | Индикатор: DONE")

    def on_enter_error(self):
        print("  [ВЫХОД] Клапан: ВЫКЛ | Барабан: СТОП | Насос: ВЫКЛ | Индикатор: ERROR")


Экземпляр конечного автомата

In [4]:
washingMachine = WashingMachine()

Список состояний

- ожидание
- набор воды
- стирка
- полоскание
- отжим
- завершение
- ошибка

In [5]:
states = [
    'idle',
    'filling',
    'washing',
    'rinsing',
    'spinning',
    'finished',
    'error'
]

Таблица переходов

- запуск программы

  ожидание -> старт -> набор воды

- бак заполнен

  набор воды -> вода набрана -> стирка

- основная стирка завершена

  стирка -> стирка завершена -> полоскание

- полоскание завершено

  полоскание -> полоскание завершено -> отжим

- отжим завершен

  отжим -> отжим завершен -> завершение

- белье выгружено

  завершение -> выгрузка -> ожидание

- возникла неисправность

  ожидание -> ошибка -> ошибка

  набор воды -> ошибка -> ошибка

  стирка -> ошибка -> ошибка

  полоскание -> ошибка -> ошибка

  отжим -> ошибка -> ошибка

  завершение -> ошибка -> ошибка

- ошибка устранена

  ошибка -> сброс -> ожидание

In [6]:
transitions_list = [
    {'trigger': 'start', 'source': 'idle', 'dest': 'filling'},
    {'trigger': 'water_full', 'source': 'filling', 'dest': 'washing'},
    {'trigger': 'wash_done', 'source': 'washing', 'dest': 'rinsing'},
    {'trigger': 'rinse_done', 'source': 'rinsing', 'dest': 'spinning'},
    {'trigger': 'spin_done', 'source': 'spinning', 'dest': 'finished'},
    {'trigger': 'unload', 'source': 'finished', 'dest': 'idle'},
    {'trigger': 'failure', 'source': ['idle', 'filling', 'washing', 'rinsing', 'spinning', 'finished'], 'dest': 'error'},
    {'trigger': 'reset', 'source': 'error', 'dest': 'idle'}
]


Выходные сигналы

- `idle` -> клапан выключен, барабан не вращается, насос выключен, индикатор READY
- `filling` -> клапан включен, барабан не вращается, насос выключен, индикатор WORK
- `washing` -> клапан выключен, барабан в режиме стирки, насос выключен, индикатор WORK
- `rinsing` -> клапан включен, барабан в режиме полоскания, насос включен, индикатор WORK
- `spinning` -> клапан выключен, барабан в режиме отжима, насос включен, индикатор WORK
- `finished` -> все исполнительные механизмы выключены, индикатор DONE
- `error` -> все исполнительные механизмы выключены, индикатор ERROR

Инициализация автомата

In [7]:
machine = Machine(washingMachine, states=states, transitions=transitions_list, initial='idle')

washingMachine.state

'idle'

Изменения состояний

In [8]:
print('=== Обычный цикл стирки ===')

washingMachine.start()
print('После start:', washingMachine.state)

washingMachine.water_full()
print('После water_full:', washingMachine.state)

washingMachine.wash_done()
print('После wash_done:', washingMachine.state)

washingMachine.rinse_done()
print('После rinse_done:', washingMachine.state)

washingMachine.spin_done()
print('После spin_done:', washingMachine.state)

washingMachine.unload()
print('После unload:', washingMachine.state)


print('\n=== Аварийная ситуация во время стирки ===')

washingMachine.start()
print('После start:', washingMachine.state)

washingMachine.water_full()
print('После water_full:', washingMachine.state)

washingMachine.failure()
print('После failure:', washingMachine.state)

washingMachine.reset()
print('После reset:', washingMachine.state)


print('\n=== Попытка неправильного перехода ===')

try:
    washingMachine.spin_done()
except Exception as e:
    print('Ошибка перехода:', e)

print('\nИтоговое состояние:', washingMachine.state)


=== Обычный цикл стирки ===
  [ВЫХОД] Клапан: ВКЛ  | Барабан: СТОП | Насос: ВЫКЛ | Индикатор: WORK
После start: filling
  [ВЫХОД] Клапан: ВЫКЛ | Барабан: СТИРКА | Насос: ВЫКЛ | Индикатор: WORK
После water_full: washing
  [ВЫХОД] Клапан: ВКЛ  | Барабан: ПОЛОСКАНИЕ | Насос: ВКЛ | Индикатор: WORK
После wash_done: rinsing
  [ВЫХОД] Клапан: ВЫКЛ | Барабан: ОТЖИМ | Насос: ВКЛ | Индикатор: WORK
После rinse_done: spinning
  [ВЫХОД] Клапан: ВЫКЛ | Барабан: СТОП | Насос: ВЫКЛ | Индикатор: DONE
После spin_done: finished
  [ВЫХОД] Клапан: ВЫКЛ | Барабан: СТОП | Насос: ВЫКЛ | Индикатор: READY
После unload: idle

=== Аварийная ситуация во время стирки ===
  [ВЫХОД] Клапан: ВКЛ  | Барабан: СТОП | Насос: ВЫКЛ | Индикатор: WORK
После start: filling
  [ВЫХОД] Клапан: ВЫКЛ | Барабан: СТИРКА | Насос: ВЫКЛ | Индикатор: WORK
После water_full: washing
  [ВЫХОД] Клапан: ВЫКЛ | Барабан: СТОП | Насос: ВЫКЛ | Индикатор: ERROR
После failure: error
  [ВЫХОД] Клапан: ВЫКЛ | Барабан: СТОП | Насос: ВЫКЛ | Индикатор: 

Диаграмма переходов

In [9]:
from mermaid import Mermaid

diagram = """
stateDiagram-v2
    [*] --> idle
    idle --> filling: start
    filling --> washing: water_full
    washing --> rinsing: wash_done
    rinsing --> spinning: rinse_done
    spinning --> finished: spin_done
    finished --> idle: unload
    idle --> error: failure
    filling --> error: failure
    washing --> error: failure
    rinsing --> error: failure
    spinning --> error: failure
    finished --> error: failure
    error --> idle: reset
"""

Mermaid(diagram)